# Multi-Model Arena with Iterative Refinement

**By: Dario Melconian**

### Agentic Design Patterns Used:
1. **Parallelization** — Multiple LLMs answer the same question simultaneously
2. **LLM-as-Judge (Evaluation)** — Claude evaluates competitor responses
3. **Iterative Refinement / Reflection** *(new!)* — Competitors receive Claude's critique and revise their answers. Claude then re-evaluates the improved responses.

### Model Setup:
- 🏆 **Judge + Question Generator**: `claude-sonnet-4-5` — sets the question and evaluates all responses
- **Competitor 1**: `gpt-4o-mini` (OpenAI)
- **Competitor 2**: Ollama local model (free, runs on your machine)
- **Competitor 3**: Groq — free API, ultra-fast Llama inference

### Finance Domain Twist:
Claude generates a **wealth management / financial reasoning** question — ensuring the prompt is sophisticated enough to meaningfully differentiate the competitors.

In [4]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

load_dotenv(override=True)

True

In [ ]:
# --- API Key checks ---
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

print(f"OpenAI:    {'✅ ' + openai_api_key[:8] if openai_api_key else 'NOT SET'}")
print(f"Anthropic: {'✅ ' + anthropic_api_key[:7] if anthropic_api_key else 'NOT SET'}")
print(f"Groq:      {'✅ ' + groq_api_key[:8] if groq_api_key else 'NOT SET — get a free key at console.groq.com'}")

In [ ]:
# --- Clients ---
openai_client = OpenAI()
claude_client = Anthropic()

# Groq uses OpenAI-compatible API
groq_client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

# Ollama uses OpenAI-compatible API locally
ollama_client = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)

# --- Model names ---
JUDGE_MODEL = "claude-sonnet-4-5"           # Claude as judge
GPT_MODEL = "gpt-4o-mini"                   # Competitor 1
OLLAMA_MODEL = "llama3.2"                   # Competitor 2 
GROQ_MODEL = "llama-3.3-70b-versatile"      # Competitor 3 

competitors = ["GPT-4o-mini", f"Ollama ({OLLAMA_MODEL})", f"Groq ({GROQ_MODEL})"]

## Step 1 — Generate a Finance/Wealth Management Question

Claude (the judge) generates a challenging, domain-specific question. Using the strongest model here ensures the question is sophisticated enough to meaningfully differentiate competitors.

In [ ]:
request = (
    "Please come up with a challenging, nuanced question about wealth management, "
    "portfolio strategy, or financial planning that requires both quantitative reasoning "
    "and ethical judgment to answer well. "
    "Answer only with the question, no explanation."
)

response = claude_client.messages.create(
    model=JUDGE_MODEL,
    max_tokens=400,
    messages=[{"role": "user", "content": request}]
)
question = response.content[0].text

display(Markdown(f"### Question\n\n{question}"))

### Question

A 45-year-old client with $8 million in investable assets has built their wealth entirely through ownership in a private defense contracting company that manufactures components for autonomous weapons systems. They want to retire at 55, need $300,000 annually (inflation-adjusted), and express feeling "somewhat uncomfortable" with how they made their money. They're considering three options: (1) sell their stake now and accept a 30% liquidity discount plus immediate 37% tax hit to invest in an ESG portfolio averaging 6% returns, (2) hold for 10 more years with expected 12% annual growth then execute a tax-efficient transfer to ESG investments, or (3) keep the concentrated position indefinitely with 12% growth and donate 50% to humanitarian causes upon death. 

How would you advise them, and what additional questions would you need answered to reconcile the tension between their financial security, tax efficiency, portfolio risk management, timeline constraints, and the ethical dimensions of both their wealth source and the ongoing harm their continued ownership might enable?

## Step 2 — Round 1: All Competitors Answer Initially

**Parallelization pattern**: same question sent to all three LLMs.

In [ ]:
# competitor 1 - GPT-4o-mini from OpenAI
def get_openai_answer(question):
    response = openai_client.chat.completions.create(
        model=GPT_MODEL,
        messages=[{"role": "user", "content": question}]
    )
    return response.choices[0].message.content

# competitor 2 - Llama 3.2 from Ollama
def get_ollama_answer(question):
    response = ollama_client.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": question}]
    )
    return response.choices[0].message.content

# competitor 3 - Llama 3.3 from Groq
def get_groq_answer(question):
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": question}]
    )
    return response.choices[0].message.content

# store answers in a dict for easy access
print("Collecting Round 1 answers...")
answers_r1 = [
    get_openai_answer(question),
    get_ollama_answer(question),
    get_groq_answer(question),
]
print("Round 1 complete.")

Round 1 complete.


In [20]:
# Display Round 1 answers
for competitor, answer in zip(competitors, answers_r1):
    display(Markdown(f"### {competitor}\n\n{answer}\n\n---"))

### GPT-4o-mini

This client is at a crucial junction in their financial planning that incorporates not just their financial goals but also their ethical concerns related to the source of their wealth. Here’s a structured way to assess their options and advice:

### Analysis of Options:

1. **Sell Now with Immediate Tax Payment and Liquidity Discount**: 
   - **Pros**: Immediate liquidity to invest in an ESG portfolio aligns with their values and provides potential long-term growth. 
   - **Cons**: The 30% liquidity discount is substantial, and the 37% tax hit will significantly reduce the net proceeds (around $4.8 million left after taxes). At an average return of 6%, they might not be able to generate the required annual income sustainably without considerable principal erosion.

2. **Hold for 10 More Years with Expected Growth**:
   - **Pros**: Retaining ownership for 10 years could yield substantial growth (estimated to reach about $25.4 million pre-tax if the 12% growth holds), leaving a greater amount to invest later. It provides time for the company to evolve socially and strategically.
   - **Cons**: Continued ownership could exacerbate their discomfort with the ethical implications of their wealth. Their portfolio remains concentrated, exposing them to higher risk.

3. **Keep the Position Indefinitely with a Plan for Donation**:
   - **Pros**: Potential for significant growth, and the ability to support humanitarian causes through legacy planning could align with their ethical priorities. 
   - **Cons**: This plan may not address their immediate financial needs since they still need $300,000 per year for living expenses. Moreover, ethical concerns remain since continued ownership supports the industry.

### Additional Questions to Consider:
1. **Retirement Timeline**: Are they flexible on the retirement age? Would they consider a phased retirement or part-time work until their financial situation is more secure?
   
2. **Risk Tolerance**: How do they feel about market volatility? Can they stomach the swings of holding a concentrated position in a single company versus the diversification they'd achieve in an ESG portfolio?
   
3. **Investment Philosophy**: How important is the alignment of investments with their personal values? Would they be willing to consider less traditional investments that might yield higher returns or more aggressive growth strategies?
   
4. **Legacy Goals**: How important is it for them to leave a financial legacy to their heirs versus philanthropic intentions? Would they prefer to support movement towards humanitarian and ethical endeavors with their wealth actively during their lifetime, instead of posthumously?
   
5. **Tax Strategy**: Are they open to consulting with a tax advisor about ways to mitigate the tax impact of selling now versus later? Could a charitable remainder trust (CRT) or donor-advised fund (DAF) help facilitate immediate donations while deferring taxes?

### Recommendations:
Taking the above considerations into account, I would most likely suggest a balanced approach:
- **Option 2** (Hold for 10 years), but with strategic planning and allocation: In this approach, actively manage the investment with regular performance reviews and consider partial sales if financial health needs dictate. Implement charitable strategies during this period that can create social impact and align with their values, possibly by creating a fund that supports humanitarian causes immediately.
  
- **Incorporate ESG Investments**: They could establish a mixed strategy where they gradually invest a portion of their assets in ESG initiatives, even while holding their concentrated position, to begin aligning their portfolio with their values.

### Conclusion:
Ultimately, the decision hinges on personal values, how they wish to manage risk, their comfort with their financial future, and how they see their legacy. Engaging with financial, legal, and tax advisors on these questions will provide them with a clearer path that addresses both their financial goals and ethical considerations.

---

### Ollama (llama3.2)

This is a complex and nuanced situation, requiring consideration of multiple factors beyond mere financial planning. I'll provide an initial assessment, and then elaborate on areas that require further clarification.

I would advise the client to carefully consider option 2: holding for 10 more years with expected 12% annual growth, followed by a tax-efficient transfer to ESG investments. Here's why:

1. **Financial security**: By growing their wealth at an attractive rate of 12%, the client can achieve their retirement goals and maintain their desired standard of living.
2. **Tax efficiency**: Delaying taxes for 10 years minimizes the capital gains tax burden, as it allows for deferral of taxed earnings to a lower tax bracket (likely in retirement).
3. **Portfolio risk management**: By diversifying into an ESG portfolio, the client can mitigate potential downsides from ownership of autonomous weapons system components and create a more sustainable investment strategy.
4. **Timeline constraints**: The client's desire to retire at 55 and achieve their annual withdrawal goal necessitates patience, as growing the wealth enough to meet their needs requires time.

However, there are still significant concerns that require further exploration:

1. **Ethical implications of ownership**: As outlined in option 3 (donating half upon death), consider the moral dimensions of continuing to invest in autonomous weapons systems and potentially perpetuating harm through their continued ownership.
2. **Risk tolerance**: How do you feel about investing in companies involved in defense technologies, even if it's indirectly? Would a tax-efficient transition to ESG investments alleviate these concerns?
3. **Tax efficiency vs. ESG goals**: How important is the transition to an ESG portfolio? Are there ESG-focused alternatives available with comparable growth rates and tax efficiencies?
4. **Wealth philanthropy**: What specific humanitarian causes or initiatives align better with your values, and would a 50% donation structure (as outlined in option 3) provide sufficient impact?
5. **Family dynamics and stakeholder considerations**: Who else might benefit if the family continues to own the company? How will decisions regarding these holdings be made, particularly after the client's retirement?

To reconcile the tension between financial security, tax efficiency, portfolio risk management, timeline constraints, and ethical dimensions, consider the following additional questions:

1. **Conduct a thorough inventory of ESG-related concerns**: What is driving your discomfort about how you built your wealth? Are there specific components or aspects of autonomous weapons systems that are causing distress?
2. **Assess the client's personal values and priorities**: How do these align with their investment philosophy? Would this alignment have been different if they hadn't made their fortune in defense contracting?
3. **Explore alternative ESG platforms or strategies**: Are there alternative ESG-focused companies, indices, or strategies that could provide a similar social return on capital while maintaining attractive growth rates and tax efficiencies?
4. **Consult with diverse stakeholders**: Discuss the client's concerns with key family members, philanthropic advisors, or other experts to gain new perspectives and ensure alignment.
5. **Consider an independent ESG assessment**: Seek out independent, third-party evaluations of their defense contracting company and its impact on society, including potential solutions to mitigate any negative effects.

Ultimately, reconciling the tension between wealth legacy, financial security, and ethical priorities requires transparent self-reflection, ongoing education, and a willingness to adapt and evolve as circumstances change.

---

### Groq (llama-3.3-70b-versatile)

I would advise the client to take a holistic approach, considering both financial and ethical factors. Here's a step-by-step analysis of the options:

**Option 1: Sell now and invest in ESG portfolio**

* Pros: Immediate diversification, alignment with ESG values, and potential for long-term growth.
* Cons: 30% liquidity discount, 37% tax hit, and potential underperformance compared to the current investment (12% growth).
* Estimated after-tax proceeds: $8 million * (1 - 0.3) * (1 - 0.37) = $3.36 million.
* Potential annual returns: 6% * $3.36 million = $201,600, which is short of the desired $300,000 annual income.

**Option 2: Hold for 10 years and transfer to ESG investments**

* Pros: Potential for higher growth (12% annual returns), tax-efficient transfer, and alignment with ESG values.
* Cons: Continued exposure to the defense contracting industry, potential reputational risk, and uncertainty about future tax laws.
* Estimated value after 10 years: $8 million * (1 + 0.12)^10 ≈ $24.3 million.
* Potential tax-efficient transfer: considering tax laws and potential transfer strategies, the client might be able to minimize taxes and transfer a significant portion of the wealth to ESG investments.

**Option 3: Keep the concentrated position indefinitely and donate upon death**

* Pros: Potential for long-term growth (12% annual returns), and philanthropic goals can be achieved through donations.
* Cons: Continued exposure to the defense contracting industry, potential reputational risk, and the client's discomfort with the wealth source.

To reconcile the tension between financial security, tax efficiency, portfolio risk management, timeline constraints, and ethical dimensions, I would ask the following additional questions:

1. **What are the client's core values and priorities?** Are they more concerned with financial security, aligning their investments with their values, or minimizing harm caused by their wealth source?
2. **What is the client's risk tolerance?** Are they willing to take on more risk to potentially achieve higher returns, or do they prioritize preserving their wealth?
3. **What are the client's philanthropic goals?** Are they interested in donating to specific causes or organizations, and how do they want to structure their philanthropy?
4. **Are there any family or legacy considerations?** Does the client have heirs or family members who may be impacted by their decisions, and are there any specific legacy goals they want to achieve?
5. **What are the potential tax implications of each option?** A detailed analysis of tax laws, potential transfer strategies, and the impact of taxes on the client's wealth would be essential.
6. **How might the client's feelings about their wealth source evolve over time?** Will their discomfort with the defense contracting industry increase or decrease as they hold onto their investment?
7. **Are there any alternative investment options that align with the client's values and goals?** Could the client consider investing in industries or companies that promote positive social or environmental impact, while also generating returns?
8. **What is the client's timeline for retirement, and how might it impact their decision?** Are they willing to delay retirement to achieve their financial goals, or do they want to prioritize their desired retirement age?

By exploring these questions and considering the client's unique circumstances, values, and priorities, we can develop a comprehensive plan that balances financial security, tax efficiency, portfolio risk management, and ethical dimensions to achieve the client's goals and alleviate their discomfort with their wealth source.

---

## Step 3 — Claude Judges Initial Round 1 and Provides Written Feedback to each LLM

LLM-as-Judge pattern: Claude give specific, actionable critique/suggestion to each competitor.

In [21]:
# Build a combined string of all answers for the judge's critiquing step
def build_together(answers):
    together = ""
    for i, answer in enumerate(answers):
        together += f"# Response from Competitor {i+1} ({competitors[i]})\n\n{answer}\n\n"
    return together

together_r1 = build_together(answers_r1)

# --- Judge's critique and feedback ---
critique_prompt = f"""You are an expert judge evaluating responses to a wealth management question.

The question asked was:
{question}

Here are the responses from {len(competitors)} competitors:

{together_r1}

For each competitor, provide:
1. A brief critique (2-3 sentences) identifying what they did well and what was weak or missing.
2. One specific suggestion they should act on in their revised answer.

Format your response as JSON with this exact structure:
{{
  "feedback": [
    {{"competitor": 1, "critique": "...", "suggestion": "..."}},
    {{"competitor": 2, "critique": "...", "suggestion": "..."}},
    {{"competitor": 3, "critique": "...", "suggestion": "..."}}
  ]
}}

Respond with JSON only. No markdown formatting or code blocks."""

# Send critique request to judge model (Claude)
critique_response = claude_client.messages.create(
    model=JUDGE_MODEL,
    max_tokens=1500,
    messages=[{"role": "user", "content": critique_prompt}]
)

# Parse and display critique feedback
critique_raw = critique_response.content[0].text
critique_data = json.loads(critique_raw)
feedbacks = critique_data["feedback"]

print("Claude's Round 1 Feedback:\n")
for fb in feedbacks:
    print(f"Competitor {fb['competitor']} ({competitors[fb['competitor']-1]}):")
    print(f"  Critique:   {fb['critique']}")
    print(f"  Suggestion: {fb['suggestion']}\n")

Claude's Round 1 Feedback:

Competitor 1 (GPT-4o-mini):
  Critique:   Provides solid financial analysis with reasonable calculations, but the recommendation to pursue Option 2 (holding for 10 years) glosses over the moral hazard of continued ownership in weapons manufacturing when the client has expressed discomfort. The response treats ethics as a secondary consideration to be 'managed' through partial charitable giving rather than directly confronting the tension between profit maximization and values alignment.
  Suggestion: Directly address the moral dimension by calculating the potential harm-per-dollar of continued ownership versus immediate divestment, and explore hybrid solutions like establishing binding commitments to divest portions annually while implementing immediate impact-offsetting strategies, rather than defaulting to wealth maximization with ethics as an afterthought.

Competitor 2 (Ollama (llama3.2)):
  Critique:   Takes the ethical considerations most seriously and

## Step 4 — Round 2: Competitors Revise Using Judge's Feedback

**Iterative Refinement / Reflection pattern** *(agentic, not workflow type pattern)*

Each model receives Claude Judge's critique of their Round 1 initial answer and is asked to produce an improved response. This imitates how a research analyst would revise a report after peer/colleague review.

In [22]:
# --- Round 2: Competitors refine their answers based on feedback, iterative refinement ---
def build_refinement_prompt(question, original_answer, feedback):
    return (
        f"You previously answered this question:\n\n"
        f"{question}\n\n"
        f"Your original answer was:\n\n{original_answer}\n\n"
        f"An expert reviewer gave you this feedback:\n"
        f"Critique: {feedback['critique']}\n"
        f"Suggestion: {feedback['suggestion']}\n\n"
        f"Please write an improved version of your answer that addresses this feedback directly. "
        f"Be specific, clear, and thorough."
    )

print("Collecting Round 2 revised answers...")

# Generate refined answers based on feedback
answers_r2 = [
    get_openai_answer(build_refinement_prompt(question, answers_r1[0], feedbacks[0])),
    get_ollama_answer(build_refinement_prompt(question, answers_r1[1], feedbacks[1])),
    get_groq_answer(build_refinement_prompt(question, answers_r1[2], feedbacks[2])),
]

print("Round 2 complete.")

Round 2 complete.


In [23]:
# Display Round 2 revised answers
for competitor, answer in zip(competitors, answers_r2):
    display(Markdown(f"### {competitor} (Revised)\n\n{answer}\n\n---"))

### GPT-4o-mini (Revised)

This client stands at a significant crossroads, where the intersection of financial security, ethical considerations, and personal values play a critical role in decision-making. Given their discomfort with the source of their wealth and the need for a sustainable income, it is essential to address the ethical implications of continued ownership in a defense company, alongside financial planning. Here’s a more comprehensive evaluation of their options with a stronger emphasis on ethical considerations.

### Analysis of Options:

1. **Sell Now with Immediate Tax Payment and Liquidity Discount**:
   - **Pros**: Immediate liquidity provides them with an opportunity to invest in an ESG portfolio that aligns with their values. This option allows them to distance themselves from an industry they find uncomfortable.
   - **Cons**: The liquidity discount (30%) and tax impact (37%) significantly reduce their net proceeds, which might not support their annual income need sustainably. They would be left with around $4.8 million, which, with an average return of 6%, may not adequately cover their inflation-adjusted annual requirement without depleting their principal.

2. **Hold for 10 More Years with Expected Growth**:
   - **Pros**: Continued ownership allows for potential growth of their wealth (estimated around $25.4 million pre-tax) that can provide significant resources for future investments into ESG portfolios. It offers time to assess the ethical landscape of the company and perhaps push for changes internally.
   - **Cons**: Continued ownership exacerbates the ethical conflict they face. While the financial growth is appealing, the moral implications of supporting a defense industry could lead to personal dissatisfaction and stress, undermining their overall well-being.

3. **Keep the Position Indefinitely with a Plan for Donation**:
   - **Pros**: This option could ultimately lead to substantial charitable contributions to humanitarian causes, potentially positioning them as a philanthropist in their later years.
   - **Cons**: Financially, this route doesn’t address their immediate need for income. Further, maintaining support for a defense industry contradicts their expressed discomfort and could continue to cause ethical distress.

### Addressing the Ethical Dimension:
**Moral Hazard of Continued Ownership**: Given the ethical implications of benefiting from a defense company that produces autonomous weapons, it’s essential to quantify the potential harm per dollar of continued ownership. For instance, the ongoing financial support to this industry indirectly funds military operations that may cause significant human suffering. 

In considering the urgency of their values alignment with their financial decisions, the following additional questions should be addressed:

### Additional Questions to Consider:
1. **Ethical Boundaries**: What specific ethical concerns do they have regarding the industry? Are there particular actions or outcomes that would push them to divest sooner rather than later?
   
2. **Income Flexibility**: Are they open to changing their living standards temporarily to accommodate the transition into a more ethical investment strategy?
   
3. **Philanthropic Interests**: What specific humanitarian causes do they want to support? Would they be interested in actively engaging with organizations that align with their values?

4. **Impact Investment Options**: How do they feel about exploring impact investments, which may provide competitive returns while aligning with their values?

### Recommendations:
1. **Immediate Partial Divestment with ESG Investments**: I recommend a hybrid approach — initiating a partial divestment strategy where the client sells a portion (e.g., 20-30%) of their stake in the defense company now and channels those proceeds into a well-researched ESG portfolio. This allows them to reduce their moral quandary while still retaining some ownership for growth.

2. **Establish a Charitable Vehicle**: They could set up a donor-advised fund (DAF) now, funding it with part of the proceeds from the initial sale to support immediate philanthropic initiatives. This creates a clear bridge between their resources and their ethical objectives, allowing funding to flow towards causes aligned with their values while managing their potential tax liabilities.

3. **Schedule Future Looks at Ownership**: They can then reassess their remaining stake every few years, systematically reducing their ownership and investing into ESG alternatives, thus addressing the ethics of continued ownership while maintaining some financial growth.

4. **Engage in Dialogue and Advocacy**: As they navigate this transition, I encourage participation in dialogues about ethical practices within the defense industry. They could advocate for changes that turn their discomfort into a catalyst for social responsibility.

### Conclusion:
The decision at hand is not solely about financial projections but fundamentally about reconciling their wealth's source with personal and societal values. By actively pursuing a balanced strategy of partial divestment and immediate philanthropic efforts, they can address both their financial security and ethical considerations head-on. Consulting with financial, legal, and tax advisors will provide the necessary insights to navigate this transition while ensuring it aligns with their life goals and values.

---

### Ollama (llama3.2) (Revised)

I would advise the client to carefully consider option 2: holding for 10 more years with expected 12% annual growth, followed by a tax-efficient transfer to ESG investments. As previously stated, this approach offers several key benefits:

1. **Financial security**: By growing their wealth at an attractive rate of 12%, the client can achieve their retirement goals and maintain their desired standard of living.
2. **Tax efficiency**: Delaying taxes for 10 years minimizes the capital gains tax burden, as it allows for deferral of taxed earnings to a lower tax bracket (likely in retirement), with approximately 30% tax savings based on projections.

However, there are still significant concerns that require further exploration:

1. **Ethical implications of ownership**: As outlined previously, consider the moral dimensions of continuing to invest in autonomous weapons systems and potentially perpetuating harm through their continued ownership.
2. **Concentration risk**: Staying undiversified for an extended period (10 years) presents a concentration risk premium, which may result in additional volatility and reduced returns compared to diversifying into ESG investments earlier.

To better address these concerns, let's proceed with the financial analysis of option 2 and consider the following calculations:

**Financial Projections:**

Assuming $8M grows at 12% annual rate for 10 years:

Year 1-5:
$8,000,000 * (1 + 0.12)^n ≈ $11,041,440 by end Year 5

Year 6-10:
End Year 5 value * (1 + 0.12)^(n+5) ≈ $18,441,919 by end Year 10

Sustainable Withdrawal Rate Calculation:

Using the 4% rule for sustainable withdrawal rates in retirement planning, we apply it to the projected end-of-Year 10 wealth.
($18,441,919 * 0.04) ≈ $735,807 per year (projected annual expense for tax-efficient conversion)

Additional Consideration of Concentration Risk Premium:

Assuming a conservative estimate of an additional 1% - 2% annual return attributed to reduced volatility by diversifying at earlier stages (roughly equivalent to investing in ESG-focused strategies with attractive growth rates), let's apply this concentration risk premium for the extended period.

We calculate concentration impact using the "Risk Premium" methodology, applying a hypothetical 1.5% annual concentration premium:

Assuming a moderate return of 12%, adjusting by 1.5% due to concentrated risk:
12% (long-term historical returns) - 1.5% = 10.5%.

Annual growth now adjusted for 10-year concentration impact at the beginning of each decade:
Annual Return @ 8M: Starting Year 0 @ 12%

Applying historical average rate to start at full potential but adjusting downward due to increased volatility through a lower return during extended period as the risk increases.
Year 5 Return at $11,041,440 ≈9% adjusted at n = 60 months
Year 10 Return at $18,441,919 using same concentration impact @70 months and using starting Year value @ 
n for returns adjusted on historical growth of a diversified ESG Investment.

The calculation is complex to solve directly and needs iteration considering different return assumptions. However it seems more likely he would approach a sustainable rate in the later years as the returns could be more stable compared with an earlier, more volatile transition into his wealth holdings.

Assuming we accept the concentration risk premium by holding for 10 more years without diversifying the portfolio beforehand (Option 2 remains viable), let's re-evaluate if meeting the client's annual retirement needs is likely with a solid projection approach using iterative analysis:


Calculating a sustainable withdrawal rate or adjusting return assumptions for reduced concentration impact could alleviate concerns about financial security. Adjusted Return projections would provide an updated picture if diversified strategies would enhance growth to meet desired returns, potentially aligning more closely with projected income needed by the client.

Finally, address key questions:

1. **Conduct a thorough inventory of ESG-related concerns**: What is driving your discomfort about how you built your wealth? Are there specific components or aspects of autonomous weapons systems that are causing distress?
2. **Assess the client's personal values and priorities**: How do these align with their investment philosophy? Would this alignment have been different if they hadn't made their fortune in defense contracting?
3. **Explore alternative ESG platforms or strategies**: Are there alternative ESG-focused companies, indices, or strategies that could provide a similar social return on capital while maintaining attractive growth rates and tax efficiencies?

By addressing these points, the client can reconcile the tension between their financial security, tax efficiency, portfolio risk management, timeline constraints, and ethical considerations.

---

### Groq (llama-3.3-70b-versatile) (Revised)

I would advise the client to adopt a holistic approach that not only considers financial factors but also grapples with the ethical implications of their wealth source and ongoing ownership of a private defense contracting company. Given the client's expressed discomfort with how they made their money, it's essential to address the moral dimensions of their decision-making process.

Initially, let's examine the financial aspects of each option:

**Option 1: Sell now and invest in ESG portfolio**

* Pros: Immediate diversification, alignment with ESG values, and potential for long-term growth.
* Cons: 30% liquidity discount, 37% tax hit, and potential underperformance compared to the current investment (12% growth).
* Estimated after-tax proceeds: $8 million * (1 - 0.3) * (1 - 0.37) = $3.36 million.
* Potential annual returns: 6% * $3.36 million = $201,600, which is short of the desired $300,000 annual income.

As the calculations demonstrate, Option 1 falls short of meeting the client's retirement needs, making it an unviable choice.

**Option 2: Hold for 10 years and transfer to ESG investments**

* Pros: Potential for higher growth (12% annual returns), tax-efficient transfer, and alignment with ESG values.
* Cons: Continued exposure to the defense contracting industry, potential reputational risk, and uncertainty about future tax laws.

However, I would argue that Option 2 raises significant ethical concerns. Continuing to profit from autonomous weapons for another decade may exacerbate the harm caused by these systems, contradicting the client's values and potentially leading to increased moral discomfort.

**Option 3: Keep the concentrated position indefinitely and donate upon death**

* Pros: Potential for long-term growth (12% annual returns), and philanthropic goals can be achieved through donations.
* Cons: Continued exposure to the defense contracting industry, potential reputational risk, and the client's discomfort with the wealth source.

Similar to Option 2, Option 3 also raises ethical concerns, as the client would continue to benefit from the defense contracting industry indefinitely.

To address the client's financial and ethical concerns, I propose a hybrid strategy that balances faster ethical alignment with improved financial outcomes:

**Hybrid Strategy: Structured 3-5 Year Divestment Plan**

1. **Tax-loss harvesting**: Identify potential tax losses within the client's existing portfolio to offset gains from the sale of their stake in the defense contracting company.
2. **Charitable remainder trusts**: Establish a charitable remainder trust to donate a portion of the proceeds to a humanitarian cause, providing a tax deduction and income stream for the client.
3. **Negotiated installment sales**: Structure the sale of the client's stake over 3-5 years, allowing for a more gradual transition and potential tax benefits.
4. **ESG investment portfolio**: Invest the proceeds from the sale in a diversified ESG portfolio, aligning with the client's values and providing potential long-term growth.
5. **Philanthropic goals**: Allocate a portion of the proceeds to support humanitarian causes, addressing the client's desire to make a positive impact.

This hybrid strategy offers several benefits:

* **Faster ethical alignment**: The client can begin to divest from the defense contracting industry within a shorter timeframe, alleviating their moral discomfort.
* **Improved financial outcomes**: By structuring the sale over 3-5 years, the client can potentially reduce tax liabilities and increase the after-tax proceeds.
* **Increased philanthropic impact**: The charitable remainder trust and allocated proceeds can support humanitarian causes, aligning with the client's values and promoting positive social impact.

To further develop this hybrid strategy, I would ask the following questions:

1. **What is the client's desired timeline for divestment?** Are they willing to consider a 3-5 year timeline, or do they want to explore alternative timeframes?
2. **How can we optimize the tax strategy for the sale?** What tax-loss harvesting opportunities exist, and how can we structure the sale to minimize tax liabilities?
3. **What are the client's philanthropic goals, and how can we support them?** Are there specific causes or organizations the client wants to support, and how can we allocate the proceeds to achieve these goals?
4. **How can we ensure the ESG investment portfolio aligns with the client's values?** What specific ESG criteria should we prioritize, and how can we monitor the portfolio's performance and impact?
5. **What are the potential risks and challenges associated with this hybrid strategy?** How can we mitigate these risks and ensure a smooth transition for the client?

By adopting this hybrid strategy, the client can address their financial and ethical concerns, achieving a more balanced and sustainable approach to their wealth management.

---

## Step 5 — Claude's Final Judgment on Revised Answers

Claude Judge evaluates the improved Round 2 responses and produces a final ranking with reasoning.

In [26]:
together_r2 = build_together(answers_r2)

# --- Final judging --- Judge evaluates revised answers and ranks competitors
final_judge_prompt = f"""You are an expert judge in a final evaluation round.

Competitors were asked:
{question}

They each received feedback on their first attempt and have now submitted revised answers.
Evaluate their revised responses on:
- Depth and accuracy of financial reasoning
- Clarity and structure
- How well they incorporated the feedback

Revised responses:
{together_r2}

Respond with JSON only in this format:
{{
  "results": ["best competitor number", "second best", "third best"],
  "reasoning": "2-3 sentences explaining your final ranking"
}}

No markdown formatting or code blocks."""

# Send final judging request to Judge model (Claude)
final_response = claude_client.messages.create(
    model=JUDGE_MODEL,
    max_tokens=800,
    messages=[{"role": "user", "content": final_judge_prompt}]
)

# Parse and display final results
final_raw = final_response.content[0].text
final_data = json.loads(final_raw)
ranks = final_data["results"]
reasoning = final_data["reasoning"]

print("FINAL RESULTS (after refinement):\n")

# enumerate through ranks and display final placements
for index, result in enumerate(ranks):
    competitor = competitors[int(result) - 1]
    medal = ["1", "2", "3"][index]
    print(f"{medal} Rank {index+1}: {competitor}")

display(Markdown(f"\n### Judge's Reasoning: \n\n{reasoning}"))

FINAL RESULTS (after refinement):

1 Rank 1: Groq (llama-3.3-70b-versatile)
2 Rank 2: GPT-4o-mini
3 Rank 3: Ollama (llama3.2)



### Judge's Reasoning: 

Competitor 3 provides the most sophisticated analysis with concrete financial calculations, a practical hybrid strategy addressing both ethical and financial concerns, and specific implementation mechanisms like charitable remainder trusts and structured installment sales. Competitor 1 offers solid ethical reasoning and a partial divestment approach but lacks the financial depth and specificity of Competitor 3. Competitor 2's response is poorly structured with incomplete calculations, confusing mathematical sections that don't reach clear conclusions, and weaker integration of ethical considerations into actionable advice.

------------------

## Summary of Agentic Patterns Used

| Pattern | Where Used |
|---|---|
| **Parallelization** | Round 1 and Round 2 - same prompt sent to all competitors simultaneously |
| **LLM-as-Judge** | Claude evaluates responses in both rounds |
| **Iterative Refinement / Reflection** | Competitors receive written critique/suggestion and revise their answers before final judgment |

### Commercial Relevance
This pattern mirrors real workflows in **wealth management research**: a junior analyst produces a draft, a senior reviewer provides structured feedback, and the analyst submits a revised version for final evaluation. 

Automating this loop with LLMs (agents/workflow) can drastically improve output quality with minimal human intervention, proving its value in any high-stakes, accuracy-critical domain.